# Advanced Outlier Detection

## Beyond Simple Thresholds

In the first data-analysis session, we covered basic outlier detection: Z-scores, IQR, and percentile-based methods. These work well for cross-sectional data where observations are independent.

But financial time series have structure:
- Prices trend up and down over months/years
- Volatility clusters (high-volatility periods followed by more high-volatility)
- Yesterday's price tells you something about today's reasonable range

In this notebook, we'll learn techniques that account for this structure:
1. **Rolling statistics** - Compare each point to its recent history
2. **Domain-specific heuristics** - Use what we know about the data (OHLC relationships)
3. **Return-based detection** - Look at changes rather than levels
4. **Handling detected outliers** - Strategies for what to do once you find them

**Optional reference.** See the [lesson guide](../readme.md) for related notebooks.
 Run from a fresh kernel. The setup below locates the lesson folder from the repository root or either notebook folder.

In [ ]:
from pathlib import Path
import os
course_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').is_file())
lesson_dir = course_root / 'Lectures' / 'Lecture 4'
os.chdir(lesson_dir)
(lesson_dir / 'outputs').mkdir(exist_ok=True)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sqlite3

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

np.random.seed(42)

## Load Data from Database

We'll use the stock data from our SQLite database and inject some artificial outliers to demonstrate detection techniques.

In [ ]:
# Connect to database
database = lesson_dir / 'data/data.db'
if not database.is_file():
    raise FileNotFoundError(f'Download data.db from bCourses to {database}')
conn = sqlite3.connect(database.resolve().as_uri() + '?mode=ro', uri=True)

# Load AAPL data
df = pd.read_sql("""
    SELECT ts, open, high, low, close, adj_close, volume, ticker
    FROM ohlc 
    WHERE ticker = 'AAPL' 
    ORDER BY ts
""", conn)

df['ts'] = pd.to_datetime(df['ts'])
df.set_index('ts', inplace=True)

# Use recent 5 years for demonstration
df = df.loc['2019-01-01':].copy()

print(f"Loaded {len(df)} rows")
print(f"Date range: {df.index.min()} to {df.index.max()}")
df.head()

In [ ]:
# Create a copy and inject some realistic outliers
df_messy = df.copy()

# Type 1: Decimal point errors (price multiplied by 10)
decimal_errors = np.random.choice(df_messy.index, size=5, replace=False)
df_messy.loc[decimal_errors, 'close'] = df_messy.loc[decimal_errors, 'close'] * 10

# Type 2: Data entry errors (random values)
entry_errors = np.random.choice(df_messy.index, size=5, replace=False)
df_messy.loc[entry_errors, 'close'] = np.random.uniform(1, 50, size=5)

# Type 3: Negative prices (impossible)
neg_errors = np.random.choice(df_messy.index, size=3, replace=False)
df_messy.loc[neg_errors, 'close'] = -abs(df_messy.loc[neg_errors, 'close'])

print(f"Injected {len(decimal_errors) + len(entry_errors) + len(neg_errors)} outliers")
print(f"Decimal errors at: {list(decimal_errors[:3])}...")
print(f"Entry errors at: {list(entry_errors[:3])}...")
print(f"Negative prices at: {list(neg_errors)}")

In [ ]:
# Visualize the problem
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df['close'].plot(ax=axes[0], title='Original Data', alpha=0.7)
axes[0].set_ylabel('Price ($)')

df_messy['close'].plot(ax=axes[1], title='Data with Outliers', alpha=0.7)
axes[1].set_ylabel('Price ($)')

plt.tight_layout()

---

# Part 1: Rolling Statistics

The key insight: in a time series, we should compare each point to **recent** values, not all values.

A price of $50 might be fine in 2020 but suspicious in 2024 if the stock has been trading at $180.

## Rolling Z-Score

Instead of computing Z-score against the entire series, compute it against a rolling window of recent data.

In [ ]:
def rolling_zscore(series, window=20):
    """Calculate Z-score against prior observations only; leave warm-up undefined."""
    rolling_mean = series.shift(1).rolling(window, min_periods=window).mean()
    rolling_std = series.shift(1).rolling(window, min_periods=window).std()
    
    # Avoid division by zero
    rolling_std = rolling_std.replace(0, np.nan)
    
    return (series - rolling_mean) / rolling_std

# Calculate rolling Z-score
df_messy['rolling_zscore'] = rolling_zscore(df_messy['close'], window=20)

# Flag outliers (|z| > 3)
threshold = 3
df_messy['is_outlier_rolling_z'] = np.abs(df_messy['rolling_zscore']) > threshold

print(f"Detected {df_messy['is_outlier_rolling_z'].sum()} outliers with rolling Z-score")

In [ ]:
# Visualize
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Price with outliers highlighted
ax = axes[0]
ax.plot(df_messy.index, df_messy['close'], alpha=0.7, label='Close')
outlier_mask = df_messy['is_outlier_rolling_z']
ax.scatter(df_messy.index[outlier_mask], df_messy['close'][outlier_mask], 
           color='red', s=50, zorder=5, label='Detected outliers')
ax.set_ylabel('Price ($)')
ax.set_title('Close Price with Rolling Z-Score Outliers')
ax.legend()

# Z-score over time
ax = axes[1]
ax.plot(df_messy.index, df_messy['rolling_zscore'], alpha=0.7)
ax.axhline(y=threshold, color='red', linestyle='--', label=f'Threshold (+/-{threshold})')
ax.axhline(y=-threshold, color='red', linestyle='--')
ax.axhline(y=0, color='gray', linestyle='-', alpha=0.5)
ax.set_ylabel('Rolling Z-Score')
ax.set_xlabel('Date')
ax.legend()

plt.tight_layout()

### Why Rolling is Better

Let's compare global Z-score (what we learned in the first data-analysis session) vs rolling Z-score:

In [ ]:
# Global Z-score
global_mean = df_messy['close'].mean()
global_std = df_messy['close'].std()
df_messy['global_zscore'] = (df_messy['close'] - global_mean) / global_std
df_messy['is_outlier_global_z'] = np.abs(df_messy['global_zscore']) > threshold

print(f"Global Z-score detected: {df_messy['is_outlier_global_z'].sum()} outliers")
print(f"Rolling Z-score detected: {df_messy['is_outlier_rolling_z'].sum()} outliers")

# Show the difference
print("\nOutliers detected by rolling but not global:")
rolling_only = df_messy['is_outlier_rolling_z'] & ~df_messy['is_outlier_global_z']
print(df_messy.loc[rolling_only, ['close', 'global_zscore', 'rolling_zscore']].head(10))

---

# Part 2: Domain-Specific Heuristics

For OHLC data, we know certain relationships must hold:
- High >= max(Open, Close)
- Low <= min(Open, Close)
- All prices should be positive
- Volume should be non-negative

Violations of these rules are definite data errors.

In [ ]:
def check_ohlc_integrity(df):
    """Check OHLC data for impossible values."""
    issues = pd.DataFrame(index=df.index)
    issues['nonfinite'] = ~np.isfinite(df[['open', 'high', 'low', 'close', 'volume']]).all(axis=1)
    
    # Rule 1: Prices must be positive
    issues['negative_price'] = (
        (df['open'] <= 0) | 
        (df['high'] <= 0) | 
        (df['low'] <= 0) | 
        (df['close'] <= 0)
    )
    
    # Rule 2: High must be >= all other prices
    issues['high_violated'] = (
        (df['high'] < df['open']) | 
        (df['high'] < df['close']) | 
        (df['high'] < df['low'])
    )
    
    # Rule 3: Low must be <= all other prices  
    issues['low_violated'] = (
        (df['low'] > df['open']) | 
        (df['low'] > df['close']) | 
        (df['low'] > df['high'])
    )
    
    # Rule 4: Volume must be non-negative
    issues['negative_volume'] = df['volume'] < 0
    
    # Any issue
    issues['any_issue'] = issues.any(axis=1)
    
    return issues

# Check our messy data
integrity = check_ohlc_integrity(df_messy)

print("OHLC Integrity Check:")
print(integrity.sum())

In [ ]:
# Show the violations
print("\nRows with integrity issues:")
print(df_messy.loc[integrity['any_issue'], ['open', 'high', 'low', 'close', 'volume']])

---

# Part 3: Return-Based Detection

Another approach: instead of looking at price levels, look at **returns** (percentage changes).

A 50% daily move is suspicious for most stocks, regardless of the price level.

In [ ]:
# Calculate daily returns
df_messy['returns'] = df_messy['close'].pct_change(fill_method=None)

# Flag extreme returns (> 20% in absolute value is very unusual for liquid stocks)
return_threshold = 0.20  # 20%
df_messy['is_outlier_return'] = np.abs(df_messy['returns']) > return_threshold

print(f"Detected {df_messy['is_outlier_return'].sum()} days with >20% moves")
print("\nExtreme return days:")
print(df_messy.loc[df_messy['is_outlier_return'], ['close', 'returns']].head(10))

In [ ]:
# Visualize return distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of returns
ax = axes[0]
df_messy['returns'].dropna().hist(bins=100, ax=ax, alpha=0.7, edgecolor='black')
ax.axvline(x=return_threshold, color='red', linestyle='--', label=f'+/-{return_threshold*100:.0f}%')
ax.axvline(x=-return_threshold, color='red', linestyle='--')
ax.set_xlabel('Daily Return')
ax.set_ylabel('Frequency')
ax.set_title('Distribution of Daily Returns')
ax.legend()

# Returns over time with outliers
ax = axes[1]
ax.plot(df_messy.index, df_messy['returns'], alpha=0.5)
outlier_mask = df_messy['is_outlier_return']
ax.scatter(df_messy.index[outlier_mask], df_messy['returns'][outlier_mask], 
           color='red', s=50, zorder=5, label='Extreme returns')
ax.axhline(y=return_threshold, color='red', linestyle='--', alpha=0.5)
ax.axhline(y=-return_threshold, color='red', linestyle='--', alpha=0.5)
ax.set_xlabel('Date')
ax.set_ylabel('Daily Return')
ax.set_title('Daily Returns Over Time')
ax.legend()

plt.tight_layout()

### Consecutive Day Check (retrospective)

This uses the next observation and cannot be used as a signal available on the flagged day.

A retrospective diagnostic: if today has an extreme return AND tomorrow has an extreme return in the opposite direction, it's likely a data error that corrects itself.

In [ ]:
def detect_spike_and_reversal(returns, threshold=0.15):
    """Detect spikes that immediately reverse (likely data errors)."""
    # Spike: large positive return followed by large negative (or vice versa)
    next_return = returns.shift(-1)
    
    spike_up_then_down = (returns > threshold) & (next_return < -threshold)
    spike_down_then_up = (returns < -threshold) & (next_return > threshold)
    
    return spike_up_then_down | spike_down_then_up

df_messy['is_spike_reversal'] = detect_spike_and_reversal(df_messy['returns'], threshold=0.15)

print(f"Detected {df_messy['is_spike_reversal'].sum()} spike-and-reversal patterns")
print("\nThese rows:")
spike_idx = df_messy[df_messy['is_spike_reversal']].index
for idx in spike_idx[:3]:
    # Show the spike and next day
    loc = df_messy.index.get_loc(idx)
    print(df_messy.iloc[loc:loc+2][['close', 'returns']])
    print()

---

# Part 3.5: Model-Based Detection with Prophet (optional extension)

This fit uses the complete sample, including the points being assessed. It is an in-sample diagnostic, not a forecast evaluation or a point-in-time detector.

So far, our methods have been rule-based. Another approach is to use a forecasting model to predict what each value "should" be, then flag points that deviate significantly from the prediction.

## How Prophet Works

**Prophet** is a time series forecasting library developed by Meta (Taylor & Letham, 2018). It uses an additive regression model:

$$y(t) = g(t) + s(t) + h(t) + \epsilon_t$$

Where:
- **$g(t)$** = Trend function (piecewise linear or logistic growth)
- **$s(t)$** = Seasonality (Fourier series for periodic patterns)
- **$h(t)$** = Holiday/event effects
- **$\epsilon_t$** = Error term (normally distributed)

### Trend Component

Prophet models trend as a piecewise linear function with changepoints:

$$g(t) = (k + \mathbf{a}(t)^T \boldsymbol{\delta}) \cdot t + (m + \mathbf{a}(t)^T \boldsymbol{\gamma})$$

Where $k$ is the base growth rate, $\boldsymbol{\delta}$ contains rate adjustments at changepoints, and $\mathbf{a}(t)$ is an indicator function for whether changepoint $j$ has occurred by time $t$. The model automatically detects where the trend changes slope.

### Seasonality Component

Seasonality is modeled using a Fourier series:

$$s(t) = \sum_{n=1}^{N} \left( a_n \cos\left(\frac{2\pi nt}{P}\right) + b_n \sin\left(\frac{2\pi nt}{P}\right) \right)$$

Where $P$ is the period (e.g., 365.25 for yearly, 7 for weekly) and $N$ controls the flexibility (more terms = more complex seasonal shapes). The coefficients $a_n$ and $b_n$ are estimated from the data.

## Using Prophet for Outlier Detection

Prophet provides **prediction intervals** based on the uncertainty in trend and seasonality estimates. The key insight:

1. Fit the model to historical data
2. Generate predictions with confidence intervals (e.g., 95%)
3. Points outside the interval are potential outliers

The prediction interval is computed via Monte Carlo simulation:
- Sample from the posterior distribution of model parameters
- Generate many possible forecast paths
- Take quantiles (e.g., 2.5% and 97.5% for a 95% interval)

**Important caveat for stock prices**: Stock prices are not stationary and don't follow the generative model Prophet assumes. However, Prophet can still be useful for detecting points that are inconsistent with recent trends and patterns. Think of it as a sophisticated smoothing approach rather than a true forecasting model for prices.

In [ ]:
from prophet import Prophet

# Prophet requires specific column names: 'ds' (datestamp) and 'y' (value to forecast)
prophet_df = df_messy[['close']].reset_index()
prophet_df.columns = ['ds', 'y']

# Configure the model
# - yearly_seasonality: captures annual patterns (e.g., January effect, summer doldrums)
# - weekly_seasonality: captures day-of-week effects (e.g., Monday effects)
# - interval_width: width of the prediction interval (0.95 = nominal 95% coverage)
model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
    interval_width=0.95,
    changepoint_prior_scale=0.05  # Controls flexibility of trend changes (lower = smoother)
)

# Fit the model - Prophet uses Stan for Bayesian inference under the hood
model.fit(prophet_df, seed=42)

print("Prophet model fitted successfully")
print(f"Detected {len(model.changepoints)} trend changepoints")

In [ ]:
# Get predictions for the same dates we have data
forecast = model.predict(prophet_df[['ds']])

# Merge predictions back with actual values
prophet_results = prophet_df.merge(
    forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']], 
    on='ds'
)

# Flag outliers: actual values outside the prediction interval
prophet_results['is_outlier_prophet'] = (
    (prophet_results['y'] < prophet_results['yhat_lower']) | 
    (prophet_results['y'] > prophet_results['yhat_upper'])
)

print(f"Prophet detected {prophet_results['is_outlier_prophet'].sum()} outliers")
print(f"(Points outside 95% prediction interval)")

In [ ]:
# Visualize Prophet's predictions and outlier detection
fig, ax = plt.subplots(figsize=(14, 6))

# Plot the prediction interval
ax.fill_between(
    prophet_results['ds'], 
    prophet_results['yhat_lower'], 
    prophet_results['yhat_upper'],
    alpha=0.3, color='blue', label='95% Prediction Interval'
)

# Plot the predicted values
ax.plot(prophet_results['ds'], prophet_results['yhat'], 
        color='blue', linewidth=1, label='Prophet Prediction')

# Plot actual values
ax.plot(prophet_results['ds'], prophet_results['y'], 
        color='black', alpha=0.7, linewidth=0.8, label='Actual')

# Highlight outliers
outlier_mask = prophet_results['is_outlier_prophet']
ax.scatter(
    prophet_results.loc[outlier_mask, 'ds'], 
    prophet_results.loc[outlier_mask, 'y'],
    color='red', s=50, zorder=5, label='Detected Outliers'
)

ax.set_xlabel('Date')
ax.set_ylabel('Price ($)')
ax.set_title('Prophet Model-Based Outlier Detection')
ax.legend(loc='upper left')
plt.tight_layout()

In [ ]:
# Visualize the decomposed components
# This shows what Prophet learned about trend and seasonality
fig_components = model.plot_components(forecast)
plt.tight_layout()

The component plot shows:
- **Trend**: The piecewise linear trend with automatically detected changepoints
- **Weekly**: Day-of-week effects (if present in the data)
- **Yearly**: Annual seasonal pattern

Note how the trend captures the overall price movement, while seasonality captures recurring patterns. Outliers are points that deviate from this combined model.

In [ ]:
# Show the detected outliers
print("Outliers detected by Prophet:")
outlier_rows = prophet_results[prophet_results['is_outlier_prophet']].copy()
outlier_rows['deviation'] = outlier_rows['y'] - outlier_rows['yhat']
outlier_rows['deviation_pct'] = (outlier_rows['deviation'] / outlier_rows['yhat']) * 100

print(outlier_rows[['ds', 'y', 'yhat', 'yhat_lower', 'yhat_upper', 'deviation_pct']].head(10).to_string())

### Prophet for Outlier Detection: Considerations

**How Outliers Are Identified:**

An observation $y_t$ is flagged as an outlier if:
$$y_t < \hat{y}_t^{(2.5\%)} \quad \text{or} \quad y_t > \hat{y}_t^{(97.5\%)}$$

Where $\hat{y}_t^{(q)}$ is the $q$-th percentile of the predictive distribution at time $t$. The interval width (95% in this example) controls sensitivity:
- Wider interval (99%) → fewer false positives, may miss real outliers
- Narrower interval (80%) → more sensitive, more false positives

**Advantages:**
- Automatically captures trend changes via changepoint detection
- Decomposes the signal into interpretable components (trend, seasonality)
- Produces probabilistic predictions with uncertainty quantification
- Handles missing data without imputation
- Can incorporate domain knowledge (holidays, known events)

**Limitations:**
- Assumes additive errors with constant variance (homoscedasticity)
- Stock prices exhibit volatility clustering (GARCH effects) which violates this
- Legitimate price jumps (earnings, M&A) may be flagged as outliers
- Slower than rule-based methods (~seconds vs milliseconds)
- Requires sufficient history (ideally 2+ years) for reliable seasonality estimation

**When to Use:**
- Data with clear seasonal patterns (retail sales, energy consumption)
- When you need interpretable decomposition of the signal
- When you want probabilistic anomaly scores rather than binary flags
- As a complement to domain-specific rules, not a replacement

**Tuning Parameters:**
- `changepoint_prior_scale`: Higher values allow more trend flexibility (default 0.05)
- `seasonality_prior_scale`: Regularization on seasonal components (default 10)
- `interval_width`: Controls prediction interval coverage (default 0.80)
- `seasonality_mode`: 'additive' (default) or 'multiplicative' for % seasonality

In [ ]:
# Add Prophet results back to our main DataFrame for use in Part 4
df_messy['is_outlier_prophet'] = prophet_results['is_outlier_prophet'].values

---

# Part 4: Combining Methods

In practice, you'll want to combine multiple detection methods. Let's create a comprehensive outlier flag.

In [ ]:
def detect_outliers_comprehensive(df, use_prophet=False):
    """
    Comprehensive outlier detection for OHLCV data.
    Returns a DataFrame with outlier flags.
    
    Parameters:
    - df: DataFrame with OHLC columns
    - use_prophet: If True, includes Prophet-based detection (slower)
    """
    result = df.copy()
    
    # Method 1: OHLC integrity
    integrity = check_ohlc_integrity(df)
    result['outlier_integrity'] = integrity['any_issue']
    
    # Method 2: Rolling Z-score on close
    result['rolling_zscore'] = rolling_zscore(df['close'], window=20)
    result['outlier_rolling_z'] = np.abs(result['rolling_zscore']) > 3
    
    # Method 3: Extreme returns
    result['returns'] = df['close'].pct_change(fill_method=None)
    result['outlier_returns'] = np.abs(result['returns']) > 0.20
    
    # Method 4: Spike and reversal
    result['outlier_spike_reversal'] = detect_spike_and_reversal(result['returns'], threshold=0.15)
    
    # Method 5: Prophet (optional, as it's slower)
    if use_prophet and 'is_outlier_prophet' in df.columns:
        result['outlier_prophet'] = df['is_outlier_prophet']
    
    # Combined: flag if ANY method detects an outlier
    outlier_cols = ['outlier_integrity', 'outlier_rolling_z', 'outlier_returns', 'outlier_spike_reversal']
    if use_prophet and 'outlier_prophet' in result.columns:
        outlier_cols.append('outlier_prophet')
    
    result['is_outlier'] = result[outlier_cols].any(axis=1)
    
    return result

# Apply comprehensive detection (including Prophet since we already ran it)
df_analyzed = detect_outliers_comprehensive(df_messy, use_prophet=True)

print("Outlier Detection Summary:")
print(f"  Integrity violations: {df_analyzed['outlier_integrity'].sum()}")
print(f"  Rolling Z-score: {df_analyzed['outlier_rolling_z'].sum()}")
print(f"  Extreme returns: {df_analyzed['outlier_returns'].sum()}")
print(f"  Spike/reversal: {df_analyzed['outlier_spike_reversal'].sum()}")
if 'outlier_prophet' in df_analyzed.columns:
    print(f"  Prophet model: {df_analyzed['outlier_prophet'].sum()}")
print(f"  Total flagged: {df_analyzed['is_outlier'].sum()}")

---

# Part 5: Handling Detected Outliers

Once you've detected outliers, you have several options:

1. **Remove** - Drop the row entirely
2. **Replace with NaN** - Then use interpolation
3. **Use alternative data** - E.g., use adjusted close instead of close
4. **Cap/Floor** - Limit to reasonable bounds

These are retrospective method comparisons. The combined flag includes plausible extremes and uses future observations in the reversal/Prophet methods. Investigate before applying repairs to real data; do not treat every statistical flag as a bad record. Interpolation and centered medians are not point-in-time transformations.

In [ ]:
def clean_outliers(df, method='interpolate'):
    """
    Clean outliers using specified method.
    
    Methods:
    - 'drop': Remove outlier rows entirely
    - 'interpolate': Replace with linear interpolation
    - 'ffill': Replace with forward fill (last known value)
    - 'median': Replace with rolling median
    """
    if method not in {'drop', 'interpolate', 'ffill', 'median'}:
        raise ValueError('Unknown cleaning method')
    result = df.copy()
    outlier_mask = result['is_outlier']
    
    if method == 'drop':
        result = result[~outlier_mask]
        
    elif method == 'interpolate':
        # Replace outliers with NaN, then interpolate
        for col in ['open', 'high', 'low', 'close', 'adj_close']:
            result.loc[outlier_mask, col] = np.nan
        price_columns = ['open', 'high', 'low', 'close', 'adj_close']
        result[price_columns] = result[price_columns].interpolate(method='linear')
        
    elif method == 'ffill':
        for col in ['open', 'high', 'low', 'close', 'adj_close']:
            result.loc[outlier_mask, col] = np.nan
        result = result.ffill()
        
    elif method == 'median':
        # Replace with rolling median
        for col in ['open', 'high', 'low', 'close', 'adj_close']:
            rolling_med = result[col].rolling(window=5, center=True, min_periods=1).median()
            result.loc[outlier_mask, col] = rolling_med[outlier_mask]
    
    return result

# Demonstrate different methods
df_clean_interp = clean_outliers(df_analyzed, method='interpolate')
df_clean_ffill = clean_outliers(df_analyzed, method='ffill')
df_clean_median = clean_outliers(df_analyzed, method='median')

In [ ]:
# Visual comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].plot(df_analyzed['close'], alpha=0.7)
axes[0, 0].set_title('Original (with outliers)')
axes[0, 0].set_ylabel('Price ($)')

axes[0, 1].plot(df_clean_interp['close'], alpha=0.7, color='green')
axes[0, 1].set_title('Cleaned: Interpolation')
axes[0, 1].set_ylabel('Price ($)')

axes[1, 0].plot(df_clean_ffill['close'], alpha=0.7, color='orange')
axes[1, 0].set_title('Cleaned: Forward Fill')
axes[1, 0].set_ylabel('Price ($)')

axes[1, 1].plot(df_clean_median['close'], alpha=0.7, color='purple')
axes[1, 1].set_title('Cleaned: Rolling Median')
axes[1, 1].set_ylabel('Price ($)')

plt.tight_layout()

## Validation: Compare to Original

Let's verify our cleaning by comparing the cleaned data to the original (before we injected outliers).

In [ ]:
# Compare cleaned vs original
comparison = pd.DataFrame({
    'original': df['close'],
    'messy': df_messy['close'],
    'cleaned': df_clean_interp['close']
})

# Calculate errors
comparison['error_messy'] = (comparison['messy'] - comparison['original']).abs()
comparison['error_cleaned'] = (comparison['cleaned'] - comparison['original']).abs()

print("Mean Absolute Error from Original:")
print(f"  Messy data: ${comparison['error_messy'].mean():.2f}")
print(f"  Cleaned data: ${comparison['error_cleaned'].mean():.2f}")
print(f"\nImprovement: {(1 - comparison['error_cleaned'].mean() / comparison['error_messy'].mean()) * 100:.1f}%")

---

# Summary

## Outlier Detection Methods for Time Series

| Method | Best For | Pros | Cons |
|--------|----------|------|------|
| Rolling Z-score | Price level anomalies | Adapts to trends | Window size matters |
| OHLC integrity | Data entry errors | Clear consistency checks | Only catches obvious errors |
| Return threshold | Extreme moves | Simple, intuitive | May flag real events |
| Spike/reversal | Data glitches | Very specific | Misses single-day errors |
| Prophet | Trend/seasonal deviations | Captures complex patterns | Slower, may miss legitimate jumps |

## Key Takeaways

1. **Use domain knowledge** - OHLC relationships, reasonable return bounds
2. **Rolling > Global** - Compare to recent history, not all history
3. **Combine methods** - No single method catches everything
4. **Model-based vs rule-based** - Prophet captures patterns but adds complexity
5. **Choose cleaning strategy carefully** - Interpolation vs forward fill vs median depends on your use case
6. **Validate** - Compare cleaned data to known good data when possible

## Quick Reference

```python
# Rolling Z-score
rolling_mean = series.rolling(window).mean()
rolling_std = series.rolling(window).std()
rolling_z = (series - rolling_mean) / rolling_std
outliers = np.abs(rolling_z) > 3

# Return-based
returns = prices.pct_change()
outliers = np.abs(returns) > 0.20

# Prophet-based
from prophet import Prophet
model = Prophet(interval_width=0.95)
model.fit(df[['ds', 'y']])
forecast = model.predict(df[['ds']])
outliers = (df['y'] < forecast['yhat_lower']) | (df['y'] > forecast['yhat_upper'])

# Clean with interpolation
prices.loc[outliers] = np.nan
prices = prices.interpolate(method='linear')
```

In [ ]:
# Close database connection
conn.close()